# 🚀 Stack 2.9 - Kaggle Training

**Free GPU training on Kaggle**

This notebook trains a LoRA adapter for Stack 2.9 on **Qwen2.5-Coder-7B** using Kaggle's free GPU.

⏱️ **Expected runtime:** 2-4 hours
💾 **VRAM needed:** ~16GB (Kaggle P100 has 16GB)

---

**Instructions:**
1. Enable GPU: Settings → Accelerator → GPU P100
2. Run cells in order from the top
3. Model auto-downloads if not present

---

In [ ]:
# STEP 1: Check GPU
import subprocess
subprocess.run(["nvidia-smi"], check=True)
print("✅ GPU ready!")

In [ ]:
# STEP 2: Clone repo and setup paths
import os
import shutil
import subprocess

# Change to a valid directory first (in case we're in a deleted folder)
os.chdir("/kaggle/working")

REPO_DIR = "/kaggle/working/stack-2.9"
MODEL_DIR = os.path.join(REPO_DIR, "base_model_qwen7b")
OUTPUT_DIR = os.path.join(REPO_DIR, "training_output")

# Remove old repo if exists (force fresh clone)
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

# Clone fresh (now includes the input_path fix)
subprocess.run(["git", "clone", "https://github.com/my-ai-stack/stack-2.9.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)

print(f"✅ Working in: {os.getcwd()}")
print(f"   MODEL_DIR: {MODEL_DIR}")
print(f"   OUTPUT_DIR: {OUTPUT_DIR}")

In [ ]:
# STEP 3: Install dependencies
import subprocess

subprocess.run(["pip", "install", "-q", "torch", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu118"], check=True)
subprocess.run(["pip", "install", "-q", "transformers", "peft", "accelerate", "datasets", "pyyaml", "tqdm", "scipy", "bitsandbytes"], check=True)
print("✅ Dependencies installed")

In [ ]:
# STEP 4: Prepare training data
import os

# Check what training data is available
REPO_TRAIN_DATA = os.path.join(REPO_DIR, "training-data/final/train.jsonl")
MINI_DATA_DIR = os.path.join(REPO_DIR, "data_mini")
MINI_DATA_FILE = os.path.join(MINI_DATA_DIR, "train_mini.jsonl")

print("🔍 Checking for training data...")
if os.path.exists(REPO_TRAIN_DATA):
    print(f"   Found full dataset: {REPO_TRAIN_DATA}")
    # Create mini subset (1K samples) for faster training
    os.makedirs(MINI_DATA_DIR, exist_ok=True)
    if not os.path.exists(MINI_DATA_FILE):
        print("   Creating mini dataset (1000 samples)...")
        import subprocess
        subprocess.run(["python", os.path.join(REPO_DIR, "scripts/create_mini_dataset.py"),
                       "--size", "1000", "--output", MINI_DATA_FILE, "--source", REPO_TRAIN_DATA], check=True)
    DATA_FILE = MINI_DATA_FILE
else:
    print("   Full dataset not found, checking for existing mini dataset...")
    if os.path.exists(MINI_DATA_FILE):
        DATA_FILE = MINI_DATA_FILE
        print(f"   Using existing mini dataset: {MINI_DATA_FILE}")
    else:
        raise FileNotFoundError("No training data found! Please ensure training-data/final/train.jsonl exists in the repo.")

print(f"\n✅ Using training data: {DATA_FILE}")
print(f"   Size: {os.path.getsize(DATA_FILE) / 1024:.1f} KB")

In [ ]:
# STEP 5: Prepare config for training
import yaml
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

config = {
    'model': {
        'name': 'Qwen/Qwen2.5-Coder-7B',
        'trust_remote_code': True,
        'torch_dtype': 'float16'
    },
    'data': {
        'train_file': DATA_FILE,  # USE THE ACTUAL DATA FILE PATH
        'max_length': 2048,
        'train_split': 1.0  # Use all data for training
    },
    'lora': {
        'r': 16,
        'alpha': 32,
        'dropout': 0.05,
        'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
        'bias': 'none',
        'task_type': 'CAUSAL_LM'
    },
    'training': {
        'num_epochs': 1,
        'batch_size': 2,
        'gradient_accumulation': 4,
        'learning_rate': 2e-4,
        'warmup_steps': 50,
        'weight_decay': 0.01,
        'max_grad_norm': 1.0,
        'logging_steps': 10,
        'save_steps': 100,
        'save_total_limit': 2,
        'fp16': True,
        'bf16': False,
        'gradient_checkpointing': True
    },
    'output': {
        'lora_dir': os.path.join(OUTPUT_DIR, 'lora'),
        'logging_dir': os.path.join(OUTPUT_DIR, 'logs')
    },
    'quantization': {
        'enabled': False
    },
    'hardware': {
        'device': 'cuda',
        'num_gpus': 1,
        'use_4bit': False,
        'use_8bit': False
    }
}

config_path = os.path.join(OUTPUT_DIR, "train_config.yaml")
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"✅ Config saved to: {config_path}")
print("\nConfig summary:")
print(f"   Model: {config['model']['name']}")
print(f"   Data: {config['data']['train_file']}")
print(f"   LoRA rank: {config['lora']['r']}")
print(f"   Batch size: {config['training']['batch_size']}")
print(f"   Epochs: {config['training']['num_epochs']}")

In [ ]:
# STEP 6: Train LoRA
import sys
sys.path.insert(0, os.path.join(REPO_DIR, "stack_2_9_training"))

print("="*60)
print("STARTING TRAINING")
print("="*60)
print(f"Config: {config_path}")
print(f"Checkpoint dir: {config['output']['lora_dir']}")
print("="*60 + "\n")

# Import and run training
from stack_2_9_training.train_lora import train_lora

try:
    trainer = train_lora(config_path)
    print("\n" + "="*60)
    print("TRAINING COMPLETED SUCCESSFULLY")
    print("="*60)
except Exception as e:
    print(f"\n❌ Training failed: {e}")
    import traceback
    traceback.print_exc()
    raise

In [ ]:
# STEP 7: Merge LoRA adapter with base model
import sys
sys.path.insert(0, os.path.join(REPO_DIR, "stack_2_9_training"))
from stack_2_9_training.merge_adapter import merge_adapter

lora_dir = config['output']['lora_dir']
merged_dir = os.path.join(OUTPUT_DIR, 'merged')
os.makedirs(merged_dir, exist_ok=True)

print("="*60)
print("MERGING LORA ADAPTER")
print("="*60)
print(f"LoRA adapter: {lora_dir}")
print(f"Output: {merged_dir}")

try:
    merge_adapter(
        base_model_name_or_path=config['model']['name'],
        adapter_path=lora_dir,
        output_path=merged_dir,
        use_safetensors=True
    )
    print("\n✅ Merge completed!")
    print(f"Merged model files: {os.listdir(merged_dir)}")
except Exception as e:
    print(f"\n❌ Merge failed: {e}")
    import traceback
    traceback.print_exc()
    raise

print("="*60)
print("🎉 ALL DONE!")
print("="*60)
print(f"\n📦 Merged model ready at: {merged_dir}")
print("\n⏳ Download the 'merged' folder from Kaggle's Output panel before the session ends!")